**Vitals BigQuery**

In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

# Using the absolute path to prevent FileNotFoundError
SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Vitals"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [
    # Primary Key & References
    bigquery.SchemaField("Vitals_Key", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("Visit_Key_B", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("Patient_U_ID", "STRING", mode="REQUIRED"),

    # Date
    bigquery.SchemaField("Visit_Date", "DATE"),

    # Vital Signs (Integers)
    bigquery.SchemaField("Systolic_BP", "INTEGER"),
    bigquery.SchemaField("Diastolic_BP", "INTEGER"),
    bigquery.SchemaField("Heart_Rate", "INTEGER"),
    bigquery.SchemaField("Pulse", "INTEGER"),
    bigquery.SchemaField("Resp_Rate", "INTEGER"),
    bigquery.SchemaField("O2_Sat", "INTEGER"),
    bigquery.SchemaField("RBS", "INTEGER"),

    # Vital Signs (Floats)
    bigquery.SchemaField("Temp", "FLOAT"),
    bigquery.SchemaField("Wt", "FLOAT")
]

# =========================================
# CREATE TABLE OBJECT
# =========================================

table = bigquery.Table(TABLE_REF, schema=schema)

# =========================================
# CLUSTERING
# =========================================

table.clustering_fields = [
    "Patient_U_ID",
    "Visit_Date"
]

# =========================================
# CREATE TABLE
# =========================================

table = client.create_table(table, exists_ok=True)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.Vitals


**Vitals Postgres**

In [15]:
import sys
import psycopg2
sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================


cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS vitals_no_seq;

""")

# =========================================
# CREATE TABLE QUERY
# =========================================

create_table_query = """
CREATE TABLE Vitals (

    -- Auto Generated Primary Key
    Vitals_Key TEXT PRIMARY KEY 
    DEFAULT (
    'VK' || 
    LPAD(nextval('vitals_no_seq')::TEXT, 3, '0')),

    -- Visit References
    Visit_Key_B TEXT NOT NULL,
    Patient_U_ID TEXT NOT NULL,
    visit_date DATE,

    -- Vital Signs (Integers)
    Systolic_BP INT,
    Diastolic_BP INT,
    Heart_Rate INT,
    Pulse INT,
    Resp_Rate INT,
    O2_Sat INT,
    RBS INT,

    -- Vital Signs (Floats/Decimals)
    Temp NUMERIC(4,1),
    Wt NUMERIC(5,1)
);
"""

# =========================================
# EXECUTE QUERIES
# =========================================

cursor.execute(create_table_query)
conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: vitals")
print("=================================")

# =========================================
# CLOSE CONNECTION
# =========================================

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: vitals
